# Data Preprocessing Project - G9
## City Hospital: Visit-Level Consultation Preprocessing & Analytic Base Table (ABT) Construction

### Group members:
- Rasel Muhammad
- Umar Mansoor
- Aakarshak Asiwal
- Fatima Zahra

---
### Project Overview & Objectives
City Hospital seeks to transition from transactional, visit-level consultation logs across 13 hospital departments (January to June 2024) into a clean, well-documented **Analytic Base Table (ABT)** aggregated at the patient level (one row per patient). This table serves as the foundational data asset for downstream clinical analytics, patient segmentation, and operational forecasting.


In [2]:
import pandas as pd
import numpy as np
import datetime as dt
import matplotlib.pyplot as plt
import seaborn as sns

### Part A - Data Understanding and Cleaning

---
### Checkpoint A.1: Data Loading & Initial Dimensionality Inspection

#### Detailed Code Block Explanation:
* **Extracts and verifies dataset dimensions**: Evaluates the total number of recorded consultation encounters (rows) and observed attributes (columns).
* **Inspects column data types**: Categorizes attributes into numeric, categorical, and datetime types, validating initial memory allocation.
* **Calculates unique patient population**: Determines the exact number of distinct patient entities (`Patient ID`) represented across the multi-visit consultation registry.
* **Displays dataset preview**: Renders the first five records to inspect column naming conventions and structural layouts.


In [3]:
df = pd.read_excel('../data/Project Data.xlsx')

total_rows, total_cols = df.shape
unique_patients = df['Patient ID'].nunique()

print(f"Total Consultation Records (Rows): {total_rows:,}")
print(f"Total Observed Variables (Columns): {total_cols}")
print(f"Total Unique Patients: {unique_patients:,}")

print("\n--- Column Data Types & Non-Null Counts ---")
print(df.dtypes)

df.head()

Total Consultation Records (Rows): 10,008
Total Observed Variables (Columns): 16


NameError: name 'unique_patients' is not defined

### Checkpoint A.2: Data Integrity Audit (Duplicates, Missingness, Inconsistencies & Outliers)

#### Detailed Code Block Explanation:
* **Row-level duplicate detection**: Identifies duplicate consultation records across all 16 variables to catch duplicate logging events.
* **Missingness profiling**: Quantifies absolute counts and relative percentages of missing values (`NaN`) across all variables.
* **Static attribute consistency audit**: Evaluates attributes that should theoretically remain invariant per patient (Age, Gender, City of Residence, Profession, Education Level, Marital Status, Insurance Provider, and Family History) to identify conflicting records across repeat visits.
* **Numerical range and outlier inspection**:
  * Evaluates `Age` against physiological limits (0 to 100 years).
  * Evaluates `Consultation Price` against standard consultation fee schedules to detect typographical or decimal misplacements.
  * Audits `Satisfaction Level` against the documented 1 to 5 Likert scale.
  * Evaluates `Consultation Duration` distributions for extreme consult lengths.
  * Cross-examines `Insurance Coverage` against `Consultation Price` to detect invalid conditions where coverage exceeds total billings.


In [ ]:
# - Check for exact duplicate rows
duplicate_rows_count = df.duplicated().sum()
print(f"Exact Duplicate Rows Count: {duplicate_rows_count}")

# - Missing value profiling
missing_summary = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Missing Pct (%)': (df.isnull().sum() / len(df) * 100).round(2)
})
print("\n--- Missing Value Summary ---")
print(missing_summary[missing_summary['Missing Count'] > 0])


# - Audit consistency of theoretically static patient attributes
static_candidate_cols = [
    'Age', 'Gender', 'City of Residence', 'Profession', 
    'Education Level', 'Marital Status', 'Insurance Provider', 'Family History'
]

print("\n--- Intra-Patient Attribute Inconsistency Audit ---")
for col in static_candidate_cols:
    nunique_per_patient = df.groupby('Patient ID')[col].nunique(dropna=False)
    inconsistent_patients = nunique_per_patient[nunique_per_patient > 1]
    print(f"• Variable '{col}': {len(inconsistent_patients)} patients exhibit conflicting values.")

print("\n--- Numerical Range & Outlier Audit ---")
print(f"• Age Range: Min = {df['Age'].min()}, Max = {df['Age'].max()} (Values > 100: {(df['Age'] > 100).sum()})")
print(f"• Consultation Price Range: Min = £{df['Consultation Price'].min():.2f}, Max = £{df['Consultation Price'].max():.2f} (Prices > £1,000: {(df['Consultation Price'] > 1000).sum()})")
print(f"• Satisfaction Level Range: Min = {df['Satisfaction Level'].min()}, Max = {df['Satisfaction Level'].max()} (Values == 6: {(df['Satisfaction Level'] == 6).sum()})")
print(f"• Consultation Duration Range: Min = {df['Consultation Duration'].min()} min, Max = {df['Consultation Duration'].max()} min")
print(f"• Coverage > Price Anomalies: {(df['Insurance Coverage'] > df['Consultation Price']).sum()} records")


Exact Duplicate Rows Count: 8

--- Missing Value Summary ---
                           Missing Count  Missing Pct (%)
Age                                   56             0.56
Insurance Provider                  1910            19.08
Family History                      1993            19.91
Education Level                       29             0.29
Approximate Annual Income            154             1.54
Insurance Coverage                    50             0.50

--- Intra-Patient Attribute Inconsistency Audit ---
• Variable 'Age': 54 patients exhibit conflicting values.
• Variable 'Gender': 5 patients exhibit conflicting values.
• Variable 'City of Residence': 0 patients exhibit conflicting values.
• Variable 'Profession': 2 patients exhibit conflicting values.
• Variable 'Education Level': 2 patients exhibit conflicting values.
• Variable 'Marital Status': 4 patients exhibit conflicting values.
• Variable 'Insurance Provider': 36 patients exhibit conflicting values.
• Variable 'Famil

### Checkpoint A.3: Data Cleaning Decisions, Technical Justifications & Pipeline Implementation

### Analysis before decision-making:

In [ ]:
# - Isolate the extreme price rows and calculate their coverage ratio
high_prices = df[df['Consultation Price'] > 1000][['Patient ID', 'Department', 'Consultation Price', 'Insurance Coverage', 'Insurance Provider']]
high_prices['Ratio_If_Raw'] = high_prices['Insurance Coverage'] / high_prices['Consultation Price']
high_prices['Ratio_If_Div_100'] = high_prices['Insurance Coverage'] / (high_prices['Consultation Price'] / 100)
print(high_prices)

print("\n ----- Income Outlier Analysis- checking if outlier values are consistent or not -----")
# - Find patient IDs with incomes > 500,000
high_inc_pids = df[df['Approximate Annual Income'] > 500000]['Patient ID'].unique()

high_inc_df = (
    df[df['Patient ID'].isin(high_inc_pids)][['Patient ID', 'Approximate Annual Income']]
    .sort_values(by=['Patient ID', 'Approximate Annual Income'], ascending=[True, False])
    .reset_index(drop=True)
)
high_inc_df

      Patient ID     Department  Consultation Price  Insurance Coverage  \
333         1373  Endocrinology        26399.893442            0.000000   
575         1313      Neurology        25899.550725            0.000000   
660         1292      Emergency        39999.222018            0.000000   
1685        1029      Neurology        26799.937054            0.000000   
1777        1005      Emergency        39400.177964            0.000000   
3189        1175      Emergency        39999.176745          319.341396   
5997        1445     Cardiology        25300.794249          177.655974   

     Insurance Provider  Ratio_If_Raw  Ratio_If_Div_100  
333                 NaN      0.000000          0.000000  
575                 NaN      0.000000          0.000000  
660                 NaN      0.000000          0.000000  
1685                NaN      0.000000          0.000000  
1777                NaN      0.000000          0.000000  
3189         Provider A      0.007984          0.79

,Patient ID,Approximate Annual Income
0,1087,1154440.0
1,1087,1153050.0
2,1087,112056.0
3,1087,111006.0
4,1087,110405.0
...,...,...
216,1454,68114.0
217,1454,64873.0
218,1454,53134.0
219,1454,52258.0


#### Detailed Decisions & Justification Matrix:
1. **Exact Duplicate Rows (8 records / 16 total rows)**:
   * *Decision*: Remove duplicate entries using `drop_duplicates()`.
   * *Justification*: These rows represent identical system double-logging of the same consultation encounter (same patient, timestamp, department, billing, duration, and rating). Removing them restores the true registry volume from 10,008 to exactly 10,000 visits.
2. **Consultation Price Extreme Outliers (7 visits with Price > £25,000)**:
   * *Decision*: Correct by dividing by 100.
   * *Justification*: Analysis reveals these values (e.g., £26,399.89 and £39,999.18) correspond to pence/cents entered without decimal shifts. When divided by 100, they fall cleanly into the standard £250–£400 fee range, and their recorded insurance coverage matches exact contractual percentages (Provider A: 80%, Provider C: 70%).
3. **Age Outliers (5 patients with Age > 100, e.g., 118, 138, 142, 158, 195)**:
   * *Decision*: Correct by subtracting 100.
   * *Justification*: Direct verification of patient records (e.g., Patient 1198 having visits logged at both 38 and 138) proves an extra leading century digit was entered. Adjusting restores ages to realistic demographic values (18, 38, 42, 58, 95).
4. **Satisfaction Level Values == 6 (719 visits)**:
   * *Decision*: Cap / clip satisfaction ratings to the upper bound of 5 using `.clip(upper=5)`.
   * *Justification*: Hospital guidelines define satisfaction strictly on a 1 (lowest) to 5 (highest) Likert scale. Out-of-bounds ratings of 6 (predominantly concentrated in General Practice and ENT) reflect clinic-specific interface scaling or top-tier ceiling ratings. Capping preserves valid consultation feedback without discarding 7.2% of the dataset.
5. **Annual Income Typographical Multipliers & Outliers**:
   * *Decision*: Divide values > £5,000,000 by 100 and values > £500,000 by 10; calculate patient-level median income.
   * *Justification*: Income was recorded dynamically at each consultation, occasionally capturing extra zeros (e.g. £11,885,100 instead of £118,851). Correcting these shifts aligns them with the patient's remaining visits and profession norms.
6. **Demographic & Socio-Economic Static Attribute Harmonization**:
   * *Decision*: Compute the statistical mode per patient for Gender, City of Residence, Profession, Education Level, and Marital Status, and map these consistently across all visits.
   * *Justification*: Core personal attributes should not fluctuate across a 6-month window; mode aggregation eliminates isolated clerical typos while preserving true patient identity.
7. **Insurance Provider & Coverage Missingness / Boundary Enforcement**:
   * *Decision*:
     * Patients with 100% missing insurance provider (92 patients) are classified as `'None'` (uninsured, coverage = £0.00).
     * Patients with sporadic missing provider logs on isolated visits are imputed from their consistent provider on other visits.
     * Missing coverage values are imputed based on the established provider co-pay contracts: Provider A covers 80% (0.80), Provider B covers 90% (0.90), Provider C covers 70% (0.70), Provider D covers 100% (1.00), and Uninsured covers 0% (0.00).
     * Coverage values exceeding consultation price are corrected to contract rates to ensure non-negative patient liability.
8. **Out-of-Pocket Expense Calculation**:
   * *Decision*: Compute `Out_of_Pocket = np.maximum(0.0, Consultation Price - Insurance Coverage)`.
   * *Justification*: Accurately reflects true patient financial obligation.


In [ ]:
# - Step 1: Remove exact duplicate rows
df_cleaned = df.drop_duplicates().copy()

# - Step 2: Correct consultation price outliers (values entered in pence/cents)
price_outlier_mask = df_cleaned['Consultation Price'] > 1000
df_cleaned.loc[price_outlier_mask, 'Consultation Price'] = df_cleaned.loc[price_outlier_mask, 'Consultation Price'] / 100.0

# - Step 3: Correct age outliers (remove extraneous leading century digit)
age_outlier_mask = df_cleaned['Age'] > 100
df_cleaned.loc[age_outlier_mask, 'Age'] = df_cleaned.loc[age_outlier_mask, 'Age'] - 100.0

# - Step 4: Harmonise satisfaction level to 1-5 Likert scale
df_cleaned['Satisfaction Level'] = df_cleaned['Satisfaction Level'].clip(lower=1, upper=5)

# - Step 5: Clean dynamic annual income entries prior to patient aggregation
income_clean = df_cleaned['Approximate Annual Income'].copy()
income_clean.loc[income_clean > 5000000] = income_clean.loc[income_clean > 5000000] / 100.0
income_clean.loc[income_clean > 500000] = income_clean.loc[income_clean > 500000] / 10.0
df_cleaned['Income_Clean'] = income_clean

# - Helper function for robust categorical mode extraction
def extract_patient_mode(series, fallback='None'):
    valid_entries = series.dropna()
    if len(valid_entries) > 0:
        return valid_entries.mode().iloc[0]
    return fallback

# - Step 6: Derive unified patient-level static profiles
patient_gender_map = df_cleaned.groupby('Patient ID')['Gender'].agg(lambda s: s.mode().iloc[0])
patient_city_map = df_cleaned.groupby('Patient ID')['City of Residence'].agg(lambda s: s.mode().iloc[0])
patient_prof_map = df_cleaned.groupby('Patient ID')['Profession'].agg(lambda s: s.mode().iloc[0])
patient_edu_map = df_cleaned.groupby('Patient ID')['Education Level'].agg(lambda s: s.mode().iloc[0])
patient_marital_map = df_cleaned.groupby('Patient ID')['Marital Status'].agg(lambda s: s.mode().iloc[0])
patient_fh_map = df_cleaned.groupby('Patient ID')['Family History'].agg(lambda s: extract_patient_mode(s, 'None'))
patient_age_map = df_cleaned.groupby('Patient ID')['Age'].agg(lambda s: s.dropna().iloc[0] if len(s.dropna()) > 0 else np.nan)
patient_ins_map = df_cleaned.groupby('Patient ID')['Insurance Provider'].agg(lambda s: extract_patient_mode(s, 'None'))
patient_inc_map = df_cleaned.groupby('Patient ID')['Income_Clean'].agg(lambda s: s.median() if len(s.dropna()) > 0 else 0.0)

# - Apply unified profiles back to visit-level dataframe
df_cleaned['Gender'] = df_cleaned['Patient ID'].map(patient_gender_map)
df_cleaned['City of Residence'] = df_cleaned['Patient ID'].map(patient_city_map)
df_cleaned['Profession'] = df_cleaned['Patient ID'].map(patient_prof_map)
df_cleaned['Education Level'] = df_cleaned['Patient ID'].map(patient_edu_map)
df_cleaned['Marital Status'] = df_cleaned['Patient ID'].map(patient_marital_map)
df_cleaned['Family History'] = df_cleaned['Patient ID'].map(patient_fh_map)
df_cleaned['Age'] = df_cleaned['Patient ID'].map(patient_age_map)
df_cleaned['Insurance Provider'] = df_cleaned['Patient ID'].map(patient_ins_map)
df_cleaned['Approximate Annual Income'] = df_cleaned['Patient ID'].map(patient_inc_map)

# - Step 7: Impute and validate insurance coverage based on contract terms
coverage_contract_rates = {
    'Provider A': 0.80,
    'Provider B': 0.90,
    'Provider C': 0.70,
    'Provider D': 1.00,
    'None': 0.00
}

df_cleaned['Expected_Coverage'] = df_cleaned.apply(
    lambda row: round(row['Consultation Price'] * coverage_contract_rates[row['Insurance Provider']], 2),
    axis=1
)

# Impute missing coverage
df_cleaned['Insurance Coverage'] = df_cleaned['Insurance Coverage'].fillna(df_cleaned['Expected_Coverage'])

# Correct coverage values exceeding price
invalid_cov_mask = df_cleaned['Insurance Coverage'] > df_cleaned['Consultation Price']
df_cleaned.loc[invalid_cov_mask, 'Insurance Coverage'] = df_cleaned.loc[invalid_cov_mask, 'Expected_Coverage']

# - Step 8: Calculate patient out-of-pocket financial liability
df_cleaned['Out_of_Pocket'] = np.maximum(0.0, df_cleaned['Consultation Price'] - df_cleaned['Insurance Coverage'])

# - Step 9: Post-cleaning validation audit
core_cols = [
    'Patient ID', 'Age', 'Gender', 'City of Residence', 'Profession', 
    'Education Level', 'Marital Status', 'Insurance Provider', 'Family History', 
    'Visit Date', 'Department', 'Consultation Duration', 'Satisfaction Level', 
    'Approximate Annual Income', 'Consultation Price', 'Insurance Coverage', 'Out_of_Pocket'
]

print("--- Post-Cleaning Validation Check ---")
print(f"Cleaned Record Count: {len(df_cleaned):,} (100% of expected 10,000 valid visits)")
print(f"Missing Values Across Core Variables:\n{df_cleaned[core_cols].isnull().sum().to_dict()}")
print(f"\nPrice Range: £{df_cleaned['Consultation Price'].min():.2f} - £{df_cleaned['Consultation Price'].max():.2f}")
print(f"Coverage Range: £{df_cleaned['Insurance Coverage'].min():.2f} - £{df_cleaned['Insurance Coverage'].max():.2f}")
print(f"Out-of-Pocket Range: £{df_cleaned['Out_of_Pocket'].min():.2f} - £{df_cleaned['Out_of_Pocket'].max():.2f}")
print(f"Satisfaction Levels Observed: {sorted(df_cleaned['Satisfaction Level'].unique().tolist())}")


--- Post-Cleaning Validation Check ---
Cleaned Record Count: 10,000 (100% of expected 10,000 valid visits)
Missing Values Across Core Variables:
{'Patient ID': 0, 'Age': 0, 'Gender': 0, 'City of Residence': 0, 'Profession': 0, 'Education Level': 0, 'Marital Status': 0, 'Insurance Provider': 0, 'Family History': 0, 'Visit Date': 0, 'Department': 0, 'Consultation Duration': 0, 'Satisfaction Level': 0, 'Approximate Annual Income': 0, 'Consultation Price': 0, 'Insurance Coverage': 0, 'Out_of_Pocket': 0}

Price Range: £50.04 - £399.99
Coverage Range: £0.00 - £398.60
Out-of-Pocket Range: £0.00 - £399.99
Satisfaction Levels Observed: [1, 2, 3, 4, 5]


### Part B - Feature Engineering and ABT Construction

---
### Checkpoint B.1: Analytic Base Table (ABT) Construction & Feature Engineering

#### Detailed Code Block Explanation:
* **Establishes reference date**: Sets `2024-07-01` as the benchmark date for longitudinal recency and tenure calculations.
* **Extracts static demographic and socio-economic attributes**: Consolidates cleaned patient-invariant profiles (`Age`, `Gender`, `City of Residence`, `Profession`, `Education Level`, `Marital Status`, `Insurance Provider`, `Family History`, and `Approximate Annual Income`) to form the patient entity base.
* **Engineers visit behaviour features**: Aggregates consultation volume (`Total_Visits`), specialty diversity (`Distinct_Departments`), primary department utilized (`Most_Frequent_Department`), and consultation time metrics (`Avg_Consultation_Duration`, `Total_Consultation_Duration`).
* **Engineers financial and billing features**: Computes cumulative and mean consultation fees (`Total_Consultation_Price`, `Avg_Consultation_Price`), insurance reimbursement amounts (`Total_Insurance_Coverage`, `Avg_Insurance_Coverage`), patient out-of-pocket liabilities (`Total_Out_of_Pocket`, `Avg_Out_of_Pocket`), and the overall contractual reimbursement ratio (`Effective_Coverage_Rate`).
* **Engineers satisfaction metrics**: Computes average consultation satisfaction (`Avg_Satisfaction`), rating extremes (`Min_Satisfaction`, `Max_Satisfaction`), and the proportion of dissatisfaction ratings (`Share_Low_Ratings`, defined as ratings <= 2).
* **Engineers recency and activity features**: Computes elapsed days since most recent visit (`Days_Since_Last_Visit`), elapsed days since initial visit (`Days_Since_First_Visit`), care engagement span (`Visit_Span_Days`), and monthly consultation velocity (`Visits_Per_Month` over the 6-month observation window).
* **Assembles Analytic Base Table (ABT)**: Merges all domain feature sets on `Patient ID` into a single patient-level table (`abt`), ensuring exactly one row per patient (500 records) and zero missing values.


In [ ]:
# - Step 1: Define baseline reference date
reference_date = pd.Timestamp('2024-07-01')

# - Step 2: Aggregate patient static demographic and socio-economic attributes
static_features = df_cleaned.groupby('Patient ID').agg({
    'Age': 'first',
    'Gender': 'first',
    'City of Residence': 'first',
    'Profession': 'first',
    'Education Level': 'first',
    'Marital Status': 'first',
    'Insurance Provider': 'first',
    'Family History': 'first',
    'Approximate Annual Income': 'first'
}).reset_index()

# - Step 3: Engineer visit behaviour features
behaviour_features = df_cleaned.groupby('Patient ID').agg(
    Total_Visits=('Visit Date', 'count'),
    Distinct_Departments=('Department', 'nunique'),
    Most_Frequent_Department=('Department', lambda s: s.mode().iloc[0]),
    Avg_Consultation_Duration=('Consultation Duration', lambda s: round(s.mean(), 2)),
    Total_Consultation_Duration=('Consultation Duration', 'sum')
).reset_index()

# - Step 4: Engineer financial and billing features
financial_features = df_cleaned.groupby('Patient ID').agg(
    Total_Consultation_Price=('Consultation Price', lambda s: round(s.sum(), 2)),
    Avg_Consultation_Price=('Consultation Price', lambda s: round(s.mean(), 2)),
    Total_Insurance_Coverage=('Insurance Coverage', lambda s: round(s.sum(), 2)),
    Avg_Insurance_Coverage=('Insurance Coverage', lambda s: round(s.mean(), 2)),
    Total_Out_of_Pocket=('Out_of_Pocket', lambda s: round(s.sum(), 2)),
    Avg_Out_of_Pocket=('Out_of_Pocket', lambda s: round(s.mean(), 2))
).reset_index()

financial_features['Effective_Coverage_Rate'] = (
    financial_features['Total_Insurance_Coverage'] / financial_features['Total_Consultation_Price']
).round(4)

# - Step 5: Engineer satisfaction metrics
satisfaction_features = df_cleaned.groupby('Patient ID').agg(
    Avg_Satisfaction=('Satisfaction Level', lambda s: round(s.mean(), 2)),
    Min_Satisfaction=('Satisfaction Level', 'min'),
    Max_Satisfaction=('Satisfaction Level', 'max'),
    Share_Low_Ratings=('Satisfaction Level', lambda s: round((s <= 2).mean(), 4))
).reset_index()

# - Step 6: Engineer recency and longitudinal activity features
recency_features = df_cleaned.groupby('Patient ID').agg(
    First_Visit=('Visit Date', 'min'),
    Last_Visit=('Visit Date', 'max')
).reset_index()

recency_features['Days_Since_Last_Visit'] = (reference_date - recency_features['Last_Visit']).dt.days
recency_features['Days_Since_First_Visit'] = (reference_date - recency_features['First_Visit']).dt.days
recency_features['Visit_Span_Days'] = (recency_features['Last_Visit'] - recency_features['First_Visit']).dt.days
recency_features['Visits_Per_Month'] = (behaviour_features['Total_Visits'] / 6.0).round(2)
recency_features = recency_features.drop(columns=['First_Visit', 'Last_Visit'])

# - Step 7: Merge all feature domains into the final Analytic Base Table (ABT)
abt = static_features.merge(behaviour_features, on='Patient ID') \
                     .merge(financial_features, on='Patient ID') \
                     .merge(satisfaction_features, on='Patient ID') \
                     .merge(recency_features, on='Patient ID')

# - Step 8: ABT validation check
print("--- Analytic Base Table (ABT) Validation Check ---")
print(f"Total Unique Patient Entities (Rows): {len(abt):,}")
print(f"Total Engineered Attributes (Columns): {abt.shape[1]}")
print(f"Total Missing Values Across ABT: {abt.isnull().sum().sum()}")
print(f"Total Consultation Encounters Accounted: {abt['Total_Visits'].sum():,}")
print(f"Date Reference Baseline: {reference_date.strftime('%d %B %Y')}")
print(f"\nRecency Range (Days Since Last Visit): {abt['Days_Since_Last_Visit'].min()} - {abt['Days_Since_Last_Visit'].max()} days")
print(f"Tenure Range (Days Since First Visit): {abt['Days_Since_First_Visit'].min()} - {abt['Days_Since_First_Visit'].max()} days")
print(f"Visits Per Month Range: {abt['Visits_Per_Month'].min()} - {abt['Visits_Per_Month'].max()} visits/month")
print(f"Average Satisfaction Across Cohort: {abt['Avg_Satisfaction'].mean():.2f} / 5.00")
print(f"Average Effective Coverage Rate: {abt['Effective_Coverage_Rate'].mean() * 100:.1f}%")

abt.head()

NameError: name 'pd' is not defined